# Scenespy no Google Colab

Notebook headless para detectar cenas, dividir vídeos por intervalo e extrair rostos sem carregar a interface gráfica.

## 1. Baixar o projeto

In [ ]:
import os
import subprocess

repository = "https://github.com/guilhermejaques/scenespy.git"
project_dir = "/content/scenespy"

if os.path.isdir(os.path.join(project_dir, ".git")):
    subprocess.run(["git", "-C", project_dir, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repository, project_dir], check=True)

os.chdir(project_dir)
print(project_dir)

## 2. Instalar as dependências

O MediaPipe usado pelo pipeline de rostos requer NumPy 1.x e Protobuf 4.x. O Colab pode avisar que outros pacotes preinstalados esperam versões mais novas; esses pacotes não fazem parte do Scenespy. Use uma sessão dedicada a este notebook.

In [ ]:
%pip install -q pillow==12.1.0 numpy==1.26.4 opencv-contrib-python==4.11.0.86 av==16.1.0 scenedetect==0.6.7.1 mediapipe==0.10.21
%pip install -q --no-deps ultralytics==8.4.9

Depois da primeira instalação, selecione **Ambiente de execução → Reiniciar sessão**. Em seguida, continue pela célula abaixo. Não é necessário executar novamente a instalação durante a mesma sessão.

## 3. Carregar e validar a API

In [ ]:
import os
import sys

project_dir = "/content/scenespy"
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from scenespy import __version__
from scenespy.api import detect_scenes, extract_faces, process_video, process_videos, split_video

print(f"Scenespy {__version__} carregado sem interface gráfica.")

## 4. Escolher o acelerador

Ative uma GPU em **Ambiente de execução → Alterar o tipo de ambiente de execução**. Com GPU NVIDIA use `nvidia`; sem GPU use `cpu`. Quando o acelerador solicitado não estiver disponível, a API usa CPU com segurança.

In [ ]:
import torch

accelerator = "nvidia" if torch.cuda.is_available() else "cpu"
print(f"Acelerador: {accelerator}")

## 5A. Enviar vídeos do computador

Use esta opção para arquivos pequenos ou testes rápidos. Os arquivos enviados ficam no armazenamento temporário da sessão.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_paths = [os.path.abspath(name) for name in uploaded]
if not video_paths:
    raise ValueError("Nenhum vídeo foi enviado.")

video_path = video_paths[0]
print(video_paths)

## 5B. Usar vídeos do Google Drive

Execute esta alternativa se os vídeos estiverem no Drive. Ajuste os caminhos depois de montar a unidade.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
video_path = "/content/drive/MyDrive/videos/video.mp4"
video_paths = [video_path]
output_root = "/content/drive/MyDrive/scenespy_output"

## 6. Definir a saída

Se você não executou a opção do Drive, os resultados serão gravados temporariamente em `/content/scenespy_output`.

In [ ]:
output_root = globals().get("output_root", "/content/scenespy_output")
os.makedirs(output_root, exist_ok=True)
print(output_root)

## 7A. Detectar e cortar cenas

Sensibilidades disponíveis: `Low`, `Normal`, `High` e `Auto`.

In [ ]:
result = detect_scenes(
    video=video_path,
    output=output_root,
    sensitivity="Normal",
    accelerator=accelerator,
)
result

## 7B. Dividir por intervalo

In [ ]:
result = split_video(
    video=video_path,
    output=output_root,
    interval=10,
    accelerator=accelerator,
)
result

## 7C. Detectar e salvar rostos

Sensibilidades disponíveis: `Low`, `Normal` e `High`. O perfil `Auto` não é usado na detecção de rostos.

In [ ]:
result = extract_faces(
    video=video_path,
    output=output_root,
    sensitivity="Normal",
    accelerator=accelerator,
)
result

## 8. Processar um lote

Modos disponíveis: `scene`, `interval` e `faces`. Por padrão, um arquivo inválido é registrado no resultado e não interrompe os demais.

In [ ]:
results = process_videos(
    videos=video_paths,
    output=output_root,
    mode="faces",
    sensitivity="Normal",
    accelerator=accelerator,
    continue_on_error=True,
)
results

## 9. Baixar os resultados

Esta etapa compacta a pasta de saída. Ela é útil quando a saída está em `/content`; para uma saída no Drive, os arquivos já permanecem salvos lá.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/scenespy_results", "zip", output_root)
files.download(archive)

## API genérica

Todos os modos também podem ser chamados por `process_video`. A API valida caminhos, modos, sensibilidades, intervalos e aceleradores antes de iniciar o processamento. Mantenha `verbose=True` para receber os mesmos textos de andamento usados pelo aplicativo.

In [ ]:
result = process_video(
    video=video_path,
    output=output_root,
    mode="scene",
    sensitivity="Normal",
    accelerator=accelerator,
    interval=10,
    verbose=True,
)
result